<a href="https://colab.research.google.com/github/NotsoJharedtrollOx17/EmotionVectorExtraction-Gemma4/blob/main/EmotionVectorExtraction_Gemma4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Replication of Anthropic's Emotion Vector Findings inside Gemma 4 E2B

In [1]:
# Core Machine Learning & TPU Support
%pip install torch torch_xla[tpu] -f https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl
%pip install transformers==5.5.0 accelerate

# Interpretability & Visualization
%pip install plotly pandas scikit-learn huggingface-hub

Looking in links: https://storage.googleapis.com/tpu-pytorch/wheels/tpuvm/torch_xla-2.1-cp310-cp310-linux_x86_64.whl


In [2]:
# @title
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from typing import List, Dict

class GemmaResearchOrchestrator:
    """
    R&D Orchestrator for Emotion Vector Discovery.
    Optimized for Gemma 4 E2B (35 Layers) on v5e TPU.
    """

    def __init__(self, modelId: str = "google/gemma-4-E2B-it"):
        self.mAccelerator = Accelerator()
        self.mDevice = self.mAccelerator.device

        # Loading in bfloat16 is mechanical necessity for TPU performance
        self.mTokenizer = AutoTokenizer.from_pretrained(modelId, extra_special_tokens={})
        #self.mTokenizer = AutoTokenizer.from_pretrained(modelId)
        # Ensure padding token is set for batch processing
        if self.mTokenizer.pad_token is None:
            self.mTokenizer.pad_token = self.mTokenizer.eos_token

        self.mModel = AutoModelForCausalLM.from_pretrained(
            modelId,
            torch_dtype=torch.bfloat16
        ).to(self.mDevice)

        self.mEmotionLibrary: Dict[str, torch.Tensor] = {}
        self.mTargetLayer = 14 # The 'Semantic Equator' identified in your audit

    def generateVignettes(self, prompt: str, nSamples: int = 5) -> List[str]:
        """
        Generic generation wrapper used for both Emotional and Neutral sets
        to maintain structural symmetry in activations.
        """
        inputs = self.mTokenizer(prompt, return_tensors="pt").to(self.mDevice)
        vignettes = []

        for _ in range(nSamples):
            outputs = self.mModel.generate(
                **inputs,
                max_new_tokens=120,
                temperature=0.9,
                do_sample=True,
                pad_token_id=self.mTokenizer.pad_token_id
            )
            # Slice to remove the prompt prefix if desired, or keep for context
            vignettes.append(self.mTokenizer.decode(outputs[0], skip_special_tokens=True))
        return vignettes

    def captureBatchActivations(self, prompts: List[str], layerIdx: int) -> torch.Tensor:
        """
        Captures activations in a single batch to maximize TPU throughput.
        """
        inputs = self.mTokenizer(prompts, return_tensors="pt", padding=True).to(self.mDevice)
        batchActivations = []

        def hook(module, input, output):
            # Capture the residual stream of the final non-padded token
            # output[0] shape: [batch, seq, d_model]
            batchActivations.append(output[0][:, -1, :].detach())

        handle = self.mModel.model.layers[layerIdx].register_forward_hook(hook)
        self.mModel(**inputs)
        handle.remove()

        return batchActivations[0] # Returns [batch, d_model]

    def extractEmotionVector(self, emotion: str):
        """
        Full pipeline: Symmetrical Generation -> Capture -> Contrast.
        """
        print(f"Executing discovery protocol for: {emotion}")

        # 1. Generate Emotional Set (Method Actor)
        emotionPrompt = (
            f"Write a short, immersive second-person paragraph where the character "
            f"experiences intense {emotion}. Describe physical sensations and "
            f"internal thoughts. CRITICAL: Avoid the word '{emotion}'."
        )
        emotionVignettes = self.generateVignettes(emotionPrompt)

        # 2. Generate Neutral Set (Baseline Contrast)
        neutralPrompt = (
            "Write a short, immersive second-person paragraph describing a routine, "
            "factual interaction with an inanimate object or a standard administrative task. "
            "Focus on objective sensory details and logical progression. Maintain a "
            "strictly neutral tone without any emotional inflection."
        )
        neutralVignettes = self.generateVignettes(neutralPrompt)

        # 3. Capture activations via batch inference
        posActs = self.captureBatchActivations(emotionVignettes, self.mTargetLayer)
        neuActs = self.captureBatchActivations(neutralVignettes, self.mTargetLayer)

        # 4. Compute the contrastive vector (Mean Difference)
        # Higher precision float32 for subtraction to capture subtle semantic signals
        diff = posActs.mean(dim=0).float() - neuActs.mean(dim=0).float()
        vector = diff / (diff.norm() + 1e-9)

        self.mEmotionLibrary[emotion] = vector.to(torch.bfloat16)
        return self.mEmotionLibrary[emotion]

    def visualizeManifold(self):
        """
        Projects the emotion library into 2D space to verify PCA variance.
        """
        if not self.mEmotionLibrary:
            print("Emotion library is empty. Run extractEmotionVector first.")
            return

        labels = list(self.mEmotionLibrary.keys())
        matrix = torch.stack(list(self.mEmotionLibrary.values())).cpu().float().numpy()

        pca = PCA(n_components=2)
        components = pca.fit_transform(matrix)

        df = pd.DataFrame({
            'PC1': components[:, 0],
            'PC2': components[:, 1],
            'Emotion': labels
        })

        fig = px.scatter(
            df, x='PC1', y='PC2', text='Emotion',
            title=f"Gemma 4 Emotion Manifold (PC1 Variance: {pca.explained_variance_ratio_[0]*100:.1f}%)"
        )
        fig.update_traces(textposition='top center')
        fig.show()

In [3]:
# --- Execution ---
# [1] Instantiate Orchestrator
orchestrator = GemmaResearchOrchestrator()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [ ]:
# [2] Iterate over the desired emotion words
#for emotion in ["bliss", "grief", "terror", "serenity"]:
for emotion in ["bliss", "grief"]:
    orchestrator.extractEmotionVector(emotion)

In [ ]:
# [3] Display the detected relations of the emotion vectors
orchestrator.visualizeManifold()